# ISA synthetic body: cloud check on a bigger model

This notebook reruns the key conditions (B, C, G, O, N) from the experiment on **Gemma 3 4B**, which is too big for an 8 GB Mac but fits on a free Colab GPU.

**Before you start:** choose **Runtime > Change runtime type > T4 GPU**, then save.

Steps: clone the repo, install, log in to Hugging Face, set the layers, run, look at the results, save to Drive. Expect roughly an hour or two in total.

## 1. Get the code

The repo is private, so cloning needs a GitHub token.

1. Create a token at GitHub > Settings > Developer settings > Personal access tokens (read access to the repo is enough).
2. In Colab, click the key icon (Secrets) in the left bar, add a secret named `GITHUB_TOKEN`, and switch on notebook access.

**Alternative:** zip the repo on your computer, upload it with the Files panel (folder icon), and run the second code cell below instead.

In [ ]:
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
!git clone https://{token}@github.com/j-greig/isa-synthetic-body.git
%cd isa-synthetic-body

In [ ]:
# Alternative: only if you uploaded isa-synthetic-body.zip instead of cloning
# !unzip -q /content/isa-synthetic-body.zip -d /content
# %cd /content/isa-synthetic-body

If you prefer to paste the token directly (not recommended, as it ends up in the notebook), the command is
`!git clone https://<TOKEN>@github.com/j-greig/isa-synthetic-body.git`, followed by `%cd isa-synthetic-body`.

## 2. Install requirements

In [ ]:
!pip install -q -r requirements.txt

## 3. Hugging Face login

Gemma is gated. Accept the licence on the model page first (for example huggingface.co/google/gemma-3-4b-it), then add a Colab secret called `HF_TOKEN` (a read token from your Hugging Face settings) with notebook access switched on.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

## 4. Settings and layers

Pick the model. The scripts read their layer numbers from `isa/config.py`, and the defaults are for Gemma 3 1B (26 layers). **Gemma 3 4B has 34 layers**, so the layers must change too. There are no command-line flags for layers, so the cell below checks the real layer count and rewrites the defaults in `isa/config.py` (in this Colab copy only) in proportion to depth:

- inject layer at about 27% of depth
- probe layers at about 50% and 65%
- loop layer and SAE layer at the first probe layer

For 34 layers this gives inject 9, probes (17, 22), loop 17, SAE 17.

In [ ]:
MODEL = "google/gemma-3-4b-it"
# MODEL = "google/gemma-2-2b-it"   # smaller alternative (26 layers; SAE release below would need changing)

In [ ]:
import re
from pathlib import Path
from transformers import AutoConfig

cfg_hf = AutoConfig.from_pretrained(MODEL)
text_cfg = getattr(cfg_hf, "text_config", cfg_hf)   # Gemma 3 4B nests it under text_config
n = text_cfg.num_hidden_layers
print("Layers in", MODEL, "=", n)

inject = round(0.27 * n)
p1, p2 = round(0.50 * n), round(0.65 * n)
print(f"inject={inject}, probes=({p1}, {p2}), loop={p1}, sae={p1}")

path = Path("isa/config.py")
src = path.read_text()
def sub(pattern, repl):
    global src
    src, k = re.subn(pattern, repl, src, count=1, flags=re.M)
    assert k == 1, pattern
sub(r"^(\s*inject_layer: int = )\d+", rf"\g<1>{inject}")
sub(r"^(\s*probe_layers: tuple = )\([^)]*\)", rf"\g<1>({p1}, {p2})")
sub(r"^(\s*loop_layer: int = )\d+", rf"\g<1>{p1}")
sub(r"^(\s*sae_layer: int = )\d+", rf"\g<1>{p1}")
sub(r'^(\s*sae_release: str = )"[^"]*"', r'\g<1>"gemma-scope-2-4b-it-res"')
sub(r'^(\s*sae_id_template: str = )"[^"]*"', r'\g<1>"layer_{layer}_width_16k_l0_medium"')
path.write_text(src)
!grep -n "layer\|sae_" isa/config.py

## 5. Run the experiment

Four steps: check the environment, train the probes, run the conditions, analyse. Runs can be stopped and restarted; finished episodes are skipped.

In [ ]:
!python scripts/00_check_env.py --model_id $MODEL

In [ ]:
!python scripts/02_train_probes.py --model_id $MODEL

In [ ]:
!python scripts/04_run.py --model_id $MODEL --conditions B,C,G,O,N --repeats 2 --name cloud_4b

In [ ]:
!python scripts/05_analyse.py runs/cloud_4b.jsonl --out artifacts/analysis_4b

## 6. Look at the results

In [ ]:
from IPython.display import Markdown, Image, display
from pathlib import Path

out = Path("artifacts/analysis_4b")
display(Markdown((out / "summary.md").read_text()))
for png in sorted(out.glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))

## 7. Save to Google Drive

Colab storage is wiped when the session ends, so copy the results to Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/isa-synthetic-body
!cp -r runs artifacts /content/drive/MyDrive/isa-synthetic-body/
!ls /content/drive/MyDrive/isa-synthetic-body